In [66]:
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path=".env")
openai_api_key = os.getenv("OPENAI_API_KEY")
tavilysearch_api_key = os.getenv("TAVILYSEARCH_API_KEY")



In [67]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(api_key=openai_api_key)

Llamado normal y con prompt al modelo

In [ ]:
# Invoca pregunta normal
llm.invoke("How much a Data Scietist earn?")

AIMessage(content="The salary of a data scientist can vary depending on factors such as the individual's level of experience, education, location, and the company they work for. In general, data scientists can expect to earn between $80,000 and $160,000 annually, with senior data scientists or those working for top tech companies potentially earning even more.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 68, 'prompt_tokens': 16, 'total_tokens': 84, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DKkQTvjocT7sjvQDzEj0LHgt6sUn4', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d00f2-9a19-7710-b0ee-164223281980-0', tool_calls=[], invalid_tool

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Crea promptp para mi llm
prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un analista de mercado laboral, puedes responder a pregutas sobre el futuro de los trabajos en tecnología"),
    ("user", "{input}")

])


In [ ]:
# Encadena el prompt al resultado del llm
chain = prompt | llm

In [15]:
# Invoca al modelo pero con un prompt previo
chain.invoke( { "input": "Debería serguir estudiando IA o buscar otra profesión?" } )

AIMessage(content='La inteligencia artificial es un campo en constante crecimiento y demanda en el mercado laboral actual. Si te interesa y tienes pasión por la IA, seguir estudiando y especializándote en este campo podría ser una excelente opción. Dada la creciente importancia de la inteligencia artificial en diversos sectores, se espera que las oportunidades laborales en este campo continúen aumentando en el futuro. Sin embargo, es importante que también consideres tus propios intereses y habilidades a la hora de tomar una decisión. Si la IA no es tu pasión, podría ser beneficioso explorar otras áreas en las que te sientas más motivado y satisfecho. ¡Sigue tus intereses y elige el camino que te lleve a una carrera profesional gratificante para ti!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 175, 'prompt_tokens': 52, 'total_tokens': 227, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_token

Memoria

In [ ]:
from langchain_core.prompts import ChatMessagePromptTemplate, MessagesPlaceholder
from langchain_openai.chat_models import ChatOpenAI

model = ChatOpenAI(api_key=api_key)                    # Crea objeto modelo

# Arma plantilla de prompt
prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Eres un asistente especializado en {ability}. "
     "Puedes responder sobre {ability}"),               # Define comportamiento del modelo
    MessagesPlaceholder(variable_name="history"),       # Recibe historial
    ("human", "{input}"),                               # Mensaje del usuario
    ])

runnable = prompt | model                               # Encadena el prompt al modelo

In [25]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}

# Recibo str y devuelve objeto tipo BaseChatMessageHistory
def get_session_history(session_id: str) -> BaseChatMessageHistory: 
    if session_id not in store:                     # Existe id en store? 
        store[session_id] = ChatMessageHistory()    # No -> Crea uno y guarda en dic
    return store[session_id]                        # Devuelve historial


with_message_history = RunnableWithMessageHistory(
    runnable,                        # Modelo encadenado
    get_session_history,             # Obtiene o crea historial
    input_messages_key="input",      # Variable de entrada de mensaje
    history_messages_key="history",  # Variable de historial
)

In [26]:
with_message_history.invoke(
    {"ability": "Ciencia de datos", "input": "qué es el R cuadrado?"},
    config = {"configurable": { "session_id": "abc123" }}
)

AIMessage(content='El coeficiente de determinación, también conocido como R cuadrado (R^2), es una medida estadística que indica la proporción de la varianza de la variable dependiente que es explicada por las variables independientes en un modelo de regresión. En otras palabras, el R cuadrado proporciona una medida de cuánto se ajustan los datos al modelo de regresión, indicando qué tan bien las variables independientes pueden predecir la variable dependiente. Un valor de R cuadrado cercano a 1 indica un buen ajuste del modelo, mientras que un valor cercano a 0 indica que el modelo no explica bien la variabilidad de los datos.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 149, 'prompt_tokens': 42, 'total_tokens': 191, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider'

In [27]:
# si no cambiamos el id de sesión, si hago otra pregunta, 
# trae el contexto de la respuesta anterior

with_message_history.invoke(
    {"ability": "Ciencia de datos", "input": "No entendi"},
    config = {"configurable": { "session_id": "abc123" }}
)

AIMessage(content='Disculpa si mi explicación no fue clara. En términos simples, el coeficiente de determinación, o R cuadrado, es un número que va de 0 a 1 que indica qué tan bien las variables independientes en un modelo de regresión pueden explicar la variabilidad en la variable dependiente. Un valor de R cuadrado cercano a 1 significa que el modelo se ajusta bien a los datos y puede predecir con precisión la variable dependiente, mientras que un valor cercano a 0 significa que el modelo no explica bien la variabilidad de los datos. ¿Hay algo más en lo que pueda ayudarte?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 141, 'prompt_tokens': 202, 'total_tokens': 343, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'syst

In [28]:
# si cambio el id de sesión, no tiene contexto previo
with_message_history.invoke(
    {"ability": "Ciencia de datos", "input": "no entendi?"},
    config = {"configurable": { "session_id": "abc456" }}
)

AIMessage(content='¡Hola! ¡Disculpa si mi respuesta anterior fue confusa! Soy un asistente especializado en Ciencia de datos, lo que significa que puedo ayudarte a responder preguntas o dudas que tengas sobre este tema. ¿En qué puedo ayudarte hoy?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 38, 'total_tokens': 95, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DKlN0AOCb3iMnpFH8bRoBZgsQ9T8h', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d0129-f946-7541-90cd-19d1f81f7cbc-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 38, 'output_tokens': 57, 'total_tokens': 95, 'input_token_details': {'au

## Tools
Son extensiones que potencian el modelo. Por ejemplo, puedo buscar en wikipedia información usando su API

In [ ]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper


In [70]:
api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=100) # Devuelve mejor resultado y caracteres especificados
tool = WikipediaQueryRun(api_wrapper=api_wrapper)                            # Uso la clase para hacer la query con el API wrapper

In [71]:
print(tool.name) # Nombre
print(tool.description) # Descripción de herramienta
print(tool.args) # Argumento

wikipedia
A wrapper around Wikipedia. Useful for when you need to answer general questions about people, places, companies, facts, historical events, or other subjects. Input should be a search query.
{'query': {'description': 'query to look up on wikipedia', 'title': 'Query', 'type': 'string'}}


In [72]:
tool.run('Data Scientist')

'Page: Data science\nSummary: Data science is an interdisciplinary academic field that uses statistics'

## Agentes
Un agente es un sistema que usa uno o más modelos para actuar.

In [63]:
from langchain_community.tools.tavily_search import TavilySearchResults

In [68]:
search = TavilySearchResults(tavily_api_key=tavilysearch_api_key)

In [69]:
search.invoke('Qué temperatura hace hoy en Buenos Aires?')

[{'title': 'Clima en Buenos Aires hoy y pronóstico del tiempo a 14 días',
  'url': 'https://www.clima.com/argentina/buenos-aires/buenos-aires',
  'content': '| 20:00 | Poco nuboso 22° 13 de MAR Hoy a las 20:00    Viento  12 km/h  Ráfagas  17 km/h  Sensación térmica  22°  Lluvia  0 mm  Nieve  0 cm  Nubes  31 %  Prob. de precipitación  20 %  Radiación UV  8 Muy alta  Humedad  65 %  Presión  1012 hPa  Amanecer  06:51  Anochecer  19:17  Fase Lunar  Luna Menguante | Despejado 22° 14 de MAR Mañana a las 20:00    Viento  10 km/h  Ráfagas  15 km/h  Sensación térmica  22°  Lluvia  0 mm  Nieve  0 cm  Nubes  12 %  Prob. de precipitación  20 %  Radiación UV  6 Alta  Humedad  66 %  Presión  1008 hPa  Amanecer  06:51  Anochecer  19:15  Fase Lunar  Luna Menguante | Despejado 27° 15 de MAR Domingo a las 20:00    Viento  13 km/h  Ráfagas  19 km/h  Sensación térmica  27°  Lluvia  0 mm  Nieve  0 cm  Nubes  10 %  Prob. de precipitación  0 %  Radiación UV  8 Muy [...] | Hoy  13 MAR  28°  16° | Mañ  14 MAR 

In [73]:
tools = [search, tool] # Lista de herramientas

In [75]:
from langchain_openai import ChatOpenAI

# Instancia nuevo modelo
llm = ChatOpenAI(model="gpt-3.5-turbo-0125", temperature=0) # Temperatura menos creatividad pero menores alucinaciones

In [85]:
from langchain import hub

prompt = hub.pull("hwchase17/openai-functions-agent")
prompt.messages

ImportError: cannot import name 'hub' from 'langchain' (c:\Users\PatricioGarcia\Desktop\Pato\Programación\Proyectos\langchain\venv_langchain\Lib\site-packages\langchain\__init__.py)